# First Qwen3 page trace
Select a GPU in **Runtime → Change runtime type**, then **Run all**. This notebook is self-contained. It downloads Qwen3-0.6B and LongBench v1 directly from Hugging Face. Allow roughly 2 GB of temporary disk for downloads. No training or GitHub login is required.

One cropped 4,096-token GovReport prompt, 16-token pages, top 64 historical pages, up to 64 generated tokens. Dense prefill yields the first generated token. Each subsequent forward pass uses the selected pages plus the pinned current page to produce the next token. An EOS or the length limit ends the pilot. One row per layer per sparse decode step.

The final cells display two miss-rate curves and export the raw trace, manifest, counts and figure. This one-document replay measures replacement headroom. It cannot establish a task-wide benefit or a serving speedup.


In [ ]:
%pip -q install "transformers==4.53.3" "huggingface-hub>=0.30,<1" matplotlib


## Select historical pages during generation
For each query head and coordinate, take the larger product of the query with the page's minimum or maximum key. Sum coordinates and average query heads to form one ranking per layer. Scores use normalized, rotary-embedded Q and K from Qwen3. The selector stores min/max bounds when a historical page first becomes eligible. Selection ranks are saved before the selected tokens are put back in chronological order for attention.

The current token's page stays pinned even on its last token. It becomes eligible for ranking on the next step. The prefill uses native SDPA. The decode backend assumes one unpadded request and full-history layers.


In [ ]:
import json
import torch
from torch.nn import functional as F
from transformers import AttentionInterface


class QuestPages:
    """A reference decode backend for one unpadded request. All KV remains on the device."""

    def __init__(self, page_size, top_k):
        self.page_size = page_size
        self.top_k = top_k
        self.bounds = {}
        self.pending = {}

    def __call__(self, module, query, key, value, attention_mask, scaling, **kwargs):
        if query.shape[0] != 1 or query.shape[2] != 1:
            raise ValueError("QuestPages expects one request and one decode query")
        layer = module.layer_idx
        kv_heads, length, dim = key.shape[1:]
        # The page containing the query is pinned, including its last token.
        history_pages = (length - 1) // self.page_size
        past = self.bounds.get(layer)
        cached = 0 if past is None else past[0].shape[1]
        if history_pages > cached:
            block = key[0, :, cached*self.page_size:history_pages*self.page_size]
            block = block.reshape(kv_heads, history_pages - cached, self.page_size, dim)
            low, high = block.amin(dim=2).float(), block.amax(dim=2).float()
            if past is not None:
                low = torch.cat((past[0], low), dim=1)
                high = torch.cat((past[1], high), dim=1)
            self.bounds[layer] = low, high
        groups = query.shape[1] // kv_heads
        if history_pages:
            low, high = self.bounds[layer]
            q = query[0, :, 0].reshape(kv_heads, groups, dim).float()
            upper = (torch.einsum("hgd,hpd->hgp", q.clamp_min(0), high)
                     + torch.einsum("hgd,hpd->hgp", q.clamp_max(0), low))
            scores = upper.mean(dim=(0, 1))
            # Page IDs start in ascending order, so stable ties prefer smaller IDs.
            ranked = scores.argsort(descending=True, stable=True)[:self.top_k]
        else:
            ranked = torch.empty(0, dtype=torch.long, device=key.device)
        self.pending[layer] = ranked
        offsets = torch.arange(self.page_size, device=key.device)
        selected = (ranked.sort().values[:, None] * self.page_size + offsets).flatten()
        current = torch.arange(history_pages*self.page_size, length, device=key.device)
        positions = torch.cat((selected, current))
        sparse_k = key.index_select(2, positions).repeat_interleave(groups, dim=1)
        sparse_v = value.index_select(2, positions).repeat_interleave(groups, dim=1)
        output = F.scaled_dot_product_attention(query, sparse_k, sparse_v,
                                                dropout_p=0.0, is_causal=False, scale=scaling)
        return output.transpose(1, 2).contiguous(), None


@torch.inference_mode()
def capture(model, prompt_ids, request_id, *, page_size, top_k, max_new_tokens, eos_ids):
    if prompt_ids.ndim != 2 or prompt_ids.shape[0] != 1 or prompt_ids.shape[1] < 1:
        raise ValueError("supply one unpadded prompt")
    if max_new_tokens < 2 or page_size < 1 or top_k < 1:
        raise ValueError("invalid token or page budget")
    cfg = model.config
    backend = QuestPages(page_size, top_k)
    original = cfg._attn_implementation
    AttentionInterface.register("rank_kv_quest", backend)
    rows, generated = [], []
    try:
        cfg._attn_implementation = "sdpa"
        output = model(input_ids=prompt_ids, use_cache=True, logits_to_keep=1)
        cache = output.past_key_values
        token = output.logits[:, -1].argmax(dim=-1, keepdim=True)
        generated.append(token.item())
        cfg._attn_implementation = "rank_kv_quest"
        for step in range(max_new_tokens - 1):
            if generated[-1] in eos_ids:
                break
            backend.pending.clear()
            output = model(input_ids=token, past_key_values=cache, use_cache=True, logits_to_keep=1)
            cache = output.past_key_values
            if set(backend.pending) != set(range(cfg.num_hidden_layers)):
                raise RuntimeError("incomplete layer trace")
            rankings = torch.stack([backend.pending[i] for i in range(cfg.num_hidden_layers)]).tolist()
            expected = min(top_k, (prompt_ids.shape[1] + step) // page_size)
            for layer, ids in enumerate(rankings):
                if len(ids) != expected:
                    raise RuntimeError("incorrect number of completed pages")
                rows.append(dict(request_id=request_id, layer_id=layer, decode_step=step,
                                 ranked_page_ids=ids))
            token = output.logits[:, -1].argmax(dim=-1, keepdim=True)
            generated.append(token.item())
    finally:
        cfg._attn_implementation = original
        # The registered callable outlives this call. Release its request-specific tensors.
        backend.bounds.clear()
        backend.pending.clear()
    return rows, generated


In [ ]:
# Exercise GQA, signed query components, tie order and the pinned page on both sides of a boundary.
from types import SimpleNamespace

def check_selector():
    torch.manual_seed(7)
    q = torch.randn(1, 4, 1, 4)
    k = torch.randn(1, 2, 9, 4)
    v = torch.randn_like(k)
    module = SimpleNamespace(layer_idx=0)
    for top_k in (1, 16):
        backend = QuestPages(page_size=4, top_k=top_k)
        for length in (4, 5, 8, 9):
            out, _ = backend(module, q, k[:, :, :length], v[:, :, :length], None, scaling=0.5)
            n = (length - 1) // 4
            scores = []
            for p in range(n):
                keys = k[0, :, p*4:(p+1)*4].repeat_interleave(2, dim=0)
                # Brute force coordinatewise max(q_d * k_d), then sum d and average heads.
                scores.append((q[0] * keys).amax(dim=1).sum(-1).mean().item())
            ids = sorted(range(n), key=lambda p: -scores[p])[:top_k]
            assert backend.pending[0].tolist() == ids
            positions = [i for p in sorted(ids) for i in range(p*4, p*4+4)] + list(range(4*n, length))
            keys = k[:, :, positions].repeat_interleave(2, dim=1)
            values = v[:, :, positions].repeat_interleave(2, dim=1)
            reference = ((q @ keys.transpose(-1, -2) * 0.5).softmax(-1) @ values).transpose(1, 2)
            torch.testing.assert_close(out, reference)
    tied = QuestPages(page_size=4, top_k=1)
    tied(module, torch.zeros_like(q), k, v, None, scaling=0.5)
    assert tied.pending[0].tolist() == [0]

check_selector()
print("Selector checks passed.")


## Load one development document
Use the first GovReport document with enough Qwen tokens for the fixed-length prompt. Keep its prefix and the complete summarization instruction. Record its ID and exclude this source document from any future held-out set. This cropped prompt is for the pilot.


In [ ]:
from hashlib import sha256
from pathlib import Path
from zipfile import ZipFile
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_REV = "c1899de289a04d12100db370d81485cdf75e47ca"
DATA_ID = "zai-org/LongBench"
DATA_REV = "5e628be450b7e67fb7ae6e201bd6d8f7056f7672"
PROMPT_TOKENS = 4096
PAGE_SIZE, TOP_K, MAX_NEW = 16, 64, 64
BUDGETS = [64, 128, 256]

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV)
archive = hf_hub_download(DATA_ID, "data.zip", repo_type="dataset", revision=DATA_REV)
prefix = tokenizer.encode("Government report excerpt:\n\n", add_special_tokens=False)
suffix = tokenizer.encode("\n\nWrite a one-paragraph summary of this report.\nSummary:", add_special_tokens=False)
context_budget = PROMPT_TOKENS - len(prefix) - len(suffix)
with ZipFile(archive) as z:
    member, = [name for name in z.namelist() if Path(name).name == "gov_report.jsonl"]
    with z.open(member) as data:
        for index, line in enumerate(data):
            doc = json.loads(line)
            context = tokenizer.encode(doc["context"], add_special_tokens=False)
            if len(context) >= context_budget:
                break
        else:
            raise RuntimeError("No GovReport document is long enough")
prompt = prefix + context[:context_budget] + suffix
request_id = f"gov_report/{doc['_id']}"
assert len(prompt) == PROMPT_TOKENS
print(f"{request_id} • row {index} • {len(prompt)} prompt tokens • deterministic prefix crop")


## Generate the trace
The model keeps its entire backing KV on the GPU for this reference run. Every replay policy subsequently sees the exact same requested pages. A full-length run produces 63 sparse steps × 28 layers, since the first generated token comes from prefill.


In [ ]:
import transformers
from transformers import AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("In Colab choose Runtime → Change runtime type → a GPU, then Run all.")
device = torch.device("cuda")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cuda.matmul.allow_tf32 = False
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REV, torch_dtype=dtype, attn_implementation="sdpa").to(device).eval()
eos = model.generation_config.eos_token_id
eos_ids = {eos} if isinstance(eos, int) else set(eos or [])
rows, generated = capture(model, torch.tensor([prompt], device=device), request_id,
                          page_size=PAGE_SIZE, top_k=TOP_K, max_new_tokens=MAX_NEW, eos_ids=eos_ids)
if not rows:
    raise RuntimeError("EOS immediately after dense prefill. No decode calls were made.")
steps = len(generated) - 1
assert len(rows) == steps * model.config.num_hidden_layers
print(f"{steps} decode steps × {model.config.num_hidden_layers} layers = {len(rows)} trace rows")
print(tokenizer.decode(generated, skip_special_tokens=True))


## Replay the trace
This cell embeds the repository's replay and plotting functions so the notebook also runs before the repository is uploaded. The offline oracle can look ahead within its own request and layer. Both policies protect the entire current selection and start with an empty historical cache.


In [ ]:
"""Replay batched sparse-KV page selections. Core replay uses the standard library."""
import argparse
import csv
import json
import math
from collections import Counter, defaultdict, deque
from pathlib import Path

POLICIES = ("lru", "fifo", "rank", "score", "belady")
FREQ_WINDOW, FREQ_CAP = 8, 4


def read_trace(path):
    """Load JSONL, preserving time order independently for each request and layer."""
    streams = defaultdict(list)
    with Path(path).open(encoding="utf-8") as source:
        for line_number, line in enumerate(source, 1):
            try:
                row = json.loads(line)
                request = row["request_id"]
                layer = row["layer_id"]
                step = row["decode_step"]
                pages = row["ranked_page_ids"]
                if not isinstance(request, str) or not request.strip():
                    raise ValueError("request_id must be a nonempty string")
                if type(layer) is not int or layer < 0:
                    raise ValueError("layer_id must be a nonnegative integer")
                stream = streams[request, layer]
                if type(step) is not int or step != len(stream):
                    raise ValueError(f"expected decode_step {len(stream)} for {(request, layer)}")
                if not isinstance(pages, list) or any(type(p) is not int or p < 0 for p in pages):
                    raise ValueError("ranked_page_ids must be a list of nonnegative integers")
                if len(pages) != len(set(pages)):
                    raise ValueError("duplicate page in the same step")
                stream.append(tuple(pages))
            except (KeyError, ValueError, TypeError) as error:
                raise ValueError(f"{path}, line {line_number}: {error}") from error
    if not streams:
        raise ValueError("empty trace")
    return dict(streams)


def infer_top_k(streams):
    """K is the widest selection in the trace. Early steps can select fewer pages."""
    return max((len(pages) for steps in streams.values() for pages in steps), default=0)


def iter_replay(steps, budget, policy, *, delta=0, beta=0, top_k=None):
    """Start one empty cache and yield (hits, misses, residents) after every step.

    A step is a set of simultaneous, compulsory admissions. Every policy protects the
    whole current selection and evicts only as many unselected residents as necessary.
    Eviction keys sort ascending, so the smallest key leaves first:
      lru    (last selection step, was a hit at that step, -page ID)
      fifo   (admission step, -page ID); hits do not refresh
      rank   (-rank at last selection, then the LRU key); rank 1 is best
      score  (tau + delta*[rank <= ceil(K/4)] + beta*min(f, 4), then the LRU key),
             where f counts selections in the FREQ_WINDOW steps before the current one
      belady (-next use step, page ID); no future use counts as infinitely far
    """
    if policy not in POLICIES or type(budget) is not int or budget < 1:
        raise ValueError(f"expected policy in {POLICIES} and a positive integer budget")
    for weight in (delta, beta):
        if type(weight) not in (int, float) or not math.isfinite(weight) or weight < 0:
            raise ValueError("delta and beta must be finite nonnegative numbers")
    if top_k is None:
        top_k = max((len(pages) for pages in steps), default=0)
    if type(top_k) is not int or top_k < 0:
        raise ValueError("top_k must be a nonnegative integer")
    top_quartile = -(-top_k // 4)
    future = defaultdict(deque)
    if policy == "belady":
        for step, pages in enumerate(steps):
            for page in pages:
                future[page].append(step)
    history = [set(pages) for pages in steps]

    def frequency(page, step):
        return sum(page in history[s] for s in range(max(0, step - FREQ_WINDOW), step))

    # Larger (step, was_hit, -page_id) means more recent in our LRU convention.
    resident = {}
    admitted = {}
    last_rank = {}
    for step, pages in enumerate(steps):
        selected = set(pages)
        if len(selected) != len(pages) or len(selected) > budget:
            raise ValueError(f"duplicate selection or insufficient budget at step {step}")
        hits = selected.intersection(resident)
        missing = selected - hits
        if policy == "belady":
            for page in selected:
                future[page].popleft()
        evict_count = max(0, len(resident) + len(missing) - budget)
        if evict_count:
            candidates = resident.keys() - selected
            if policy == "lru":
                key = resident.__getitem__
            elif policy == "fifo":
                def key(page):
                    return admitted[page], -page
            elif policy == "rank":
                def key(page):
                    return (-last_rank[page],) + resident[page]
            elif policy == "score":
                def key(page):
                    bonus = delta * (last_rank[page] <= top_quartile)
                    bonus += beta * min(frequency(page, step), FREQ_CAP)
                    return (resident[page][0] + bonus,) + resident[page]
            else:
                def key(page):
                    next_step = future[page][0] if future[page] else len(steps)
                    return -next_step, page
            for page in sorted(candidates, key=key)[:evict_count]:
                del resident[page], admitted[page], last_rank[page]
        for rank, page in enumerate(pages, 1):
            resident[page] = (step, page in hits, -page)
            last_rank[page] = rank
            if page in missing:
                admitted[page] = step
        yield hits, missing, frozenset(resident)


def replay_stream(steps, budget, policy, *, delta=0, beta=0, top_k=None):
    """Count accesses and misses for one request/layer stream."""
    seen = set()
    accesses = misses = 0
    for hits, missing, _ in iter_replay(steps, budget, policy,
                                        delta=delta, beta=beta, top_k=top_k):
        accesses += len(hits) + len(missing)
        misses += len(missing)
        seen.update(hits, missing)
    return Counter(accesses=accesses, misses=misses, compulsory_misses=len(seen),
                   repeat_misses=misses - len(seen), repeat_accesses=accesses - len(seen))


def parse_policy(spec, delta=0, beta=0):
    """Map a CLI name to (label, policy, delta, beta).

    rf is recency plus frequency (delta=0), rrank is recency plus rank (beta=0) and
    full uses both weights. score:D:B names explicit weights, for example score:2:1.
    """
    fixed = {"lru": "lru", "fifo": "fifo", "rank": "rank", "belady": "belady"}
    presets = {"rf": (0, beta), "rrank": (delta, 0), "full": (delta, beta)}
    if spec in fixed:
        return spec, fixed[spec], None, None
    if spec in presets:
        return (spec, "score") + presets[spec]
    parts = spec.split(":")
    if len(parts) == 3 and parts[0] == "score":
        try:
            weights = [int(p) if p.isdigit() else float(p) for p in parts[1:]]
        except ValueError:
            weights = None
        if weights is not None:
            return (spec, "score") + tuple(weights)
    raise ValueError(f"unknown policy {spec!r}; use {', '.join([*fixed, *presets])} or score:D:B")


def compare(streams, budgets, policies=("lru", "belady"), *, delta=0, beta=0, top_k=None):
    """Pool counts across streams for each budget and policy, in the order given."""
    if top_k is None:
        top_k = infer_top_k(streams)
    specs = [parse_policy(spec, delta, beta) for spec in policies]
    rows = []
    for budget in budgets:
        for label, policy, d, b in specs:
            counts = Counter()
            for steps in streams.values():
                counts.update(replay_stream(steps, budget, policy,
                                            delta=d or 0, beta=b or 0, top_k=top_k))
            rows.append(dict(
                budget=budget, policy=label, delta=d, beta=b, top_k=top_k,
                accesses=counts["accesses"], misses=counts["misses"],
                compulsory_misses=counts["compulsory_misses"],
                repeat_misses=counts["repeat_misses"], repeat_accesses=counts["repeat_accesses"],
                miss_rate=counts["misses"] / counts["accesses"] if counts["accesses"] else None,
                repeat_miss_rate=(counts["repeat_misses"] / counts["repeat_accesses"]
                                  if counts["repeat_accesses"] else None),
            ))
    return rows


def save_comparison(rows, out_dir, title):
    """Write comparison.csv, then comparison.png when Matplotlib is installed."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with (out_dir / "comparison.csv").open("w", newline="", encoding="utf-8") as output:
        writer = csv.DictWriter(output, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Matplotlib is not installed, so comparison.png was skipped.")
        return
    names = {"lru": "LRU", "belady": "Bélády", "fifo": "FIFO", "rank": "Rank only",
             "rf": "Recency + freq.", "rrank": "Recency + rank", "full": "Full"}
    styles = {"lru": "o-", "belady": "x--"}
    labels = list(dict.fromkeys(r["policy"] for r in rows))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), layout="constrained")
    for axis, metric, label in zip(axes, ("miss_rate", "repeat_miss_rate"),
                                  ("All selected pages", "Previously selected pages")):
        for policy in labels:
            subset = sorted((r for r in rows if r["policy"] == policy), key=lambda r: r["budget"])
            rates = [float("nan") if r[metric] is None else 100 * r[metric] for r in subset]
            axis.plot([r["budget"] for r in subset], rates, styles.get(policy, "s:"),
                      label=names.get(policy, policy))
        axis.set(title=label, xlabel="Cache budget (pages per layer)", ylabel="Miss rate (%)")
        axis.set_xticks(sorted({r["budget"] for r in rows}))
        axis.set_ylim(bottom=0)
        axis.grid(axis="y", alpha=.25)
        axis.legend()
    fig.suptitle(title)
    fig.savefig(out_dir / "comparison.png", dpi=160)
    plt.close(fig)

## Compare the budgets
The left plot uses every historical page access. The right uses repeat accesses only. Each rate pools counts across layers. At 64 pages both curves must coincide. At 128 pages their vertical distance is the replacement headroom on this prompt. The CSV also includes first-touch and repeat counts. A zero gap is a valid result.


In [ ]:
from datetime import datetime, timezone
import platform
from zipfile import ZIP_DEFLATED
from IPython.display import Image, display

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = Path("results") / f"qwen3-govreport-{stamp}"
run_dir.mkdir(parents=True)
trace_path = run_dir / "trace.jsonl"
with trace_path.open("w", encoding="utf-8") as output:
    for row in rows:
        print(json.dumps(row, separators=(",", ":")), file=output)
manifest = dict(
    schema_version=1, request_id=request_id, dataset=DATA_ID, data_revision=DATA_REV,
    dataset_member=member, row_index=index, original_context_tokens=len(context),
    prompt_tokens=len(prompt), prompt_ids_sha256=sha256(json.dumps(prompt).encode()).hexdigest(),
    prompt_style="plain GovReport prefix excerpt; one-paragraph summary",
    model=MODEL_ID, model_revision=MODEL_REV, dtype=str(dtype), gpu=torch.cuda.get_device_name(0),
    python=platform.python_version(), torch=str(torch.__version__), cuda=torch.version.cuda,
    transformers=transformers.__version__, page_size=PAGE_SIZE, top_k=TOP_K, budgets=BUDGETS,
    num_layers=model.config.num_hidden_layers, decode_steps=steps, max_new_tokens=MAX_NEW,
    generation="greedy, batch=1", eos_token_ids=sorted(eos_ids), generated_token_ids=generated,
    stop_reason="eos" if generated[-1] in eos_ids else "length",
    selector="post-RoPE signed min/max bounds averaged over query heads; stable by page ID",
    trace_sha256=sha256(trace_path.read_bytes()).hexdigest(),
)
(run_dir / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

streams = read_trace(trace_path)
summary = compare(streams, BUDGETS)
assert len(streams) == model.config.num_hidden_layers
assert all(len(s) == steps for s in streams.values())
# For this 4K prompt all selections have exactly K completed pages.
k_misses = sum(len(s[0]) + sum(len(set(b) - set(a)) for a, b in zip(s, s[1:]))
               for s in streams.values())
for budget in BUDGETS:
    lru, oracle = [r for r in summary if r["budget"] == budget]
    assert lru["misses"] >= oracle["misses"]
    assert lru["compulsory_misses"] == oracle["compulsory_misses"]
    if budget == TOP_K:
        assert lru["misses"] == oracle["misses"] == k_misses
    print(f"B={budget:3d}  LRU {lru['miss_rate']:.2%}  Bélády {oracle['miss_rate']:.2%}  "
          f"repeat misses {lru['repeat_misses']} vs {oracle['repeat_misses']}  "
          f"headroom {100*(lru['miss_rate']-oracle['miss_rate']):.2f} percentage points")
save_comparison(summary, run_dir, f"Qwen3-0.6B · one GovReport · {steps} sparse decode steps")
display(Image(filename=str(run_dir / "comparison.png")))
bundle = run_dir.with_suffix(".zip")
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as z:
    for name in ("trace.jsonl", "manifest.json", "comparison.csv", "comparison.png"):
        z.write(run_dir / name, arcname=name)
print(f"Saved {bundle.resolve()}")


## Download this run
The ZIP contains the JSONL trace, a reproducibility manifest, the CSV and the PNG. If the browser blocks the download, use the Colab Files panel to download the printed ZIP path before the session ends.


In [ ]:
from google.colab import files
files.download(str(bundle))


## Sources

- [Qwen3-0.6B model](https://huggingface.co/Qwen/Qwen3-0.6B)
- [LongBench v1 dataset](https://huggingface.co/datasets/zai-org/LongBench)
- [Quest, ICML 2024](https://proceedings.mlr.press/v235/tang24l.html)
- [Pinned Transformers Qwen3 attention implementation](https://github.com/huggingface/transformers/blob/v4.53.3/src/transformers/models/qwen3/modeling_qwen3.py)
